In [ ]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(os.environ['TORCH'])

%pip install -q torch_scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
%pip install -q torch_sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
%pip install -q pyg_lib -f https://data.pyg.org/whl/torch-${TORCH}.html
%pip install -q torch_geometric -f https://data.pyg.org/whl/torch-${TORCH}.html

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [ ]:
import io, math, time, random
import polars as pl
import requests
import torch
import torch.nn.functional as F
import numpy as np
import torch.nn as nn
import torch_geometric.nn as gnn
from torch import Tensor
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
MODEL_PATH = '../models/gnn/model.pt'

In [ ]:
BASE = 'https://huggingface.co/datasets/najrum/cf-interactions/resolve/main/'

def fetch(path):
    r = requests.get(BASE + path, timeout=120); r.raise_for_status()
    return io.BytesIO(r.content)

train    = pl.read_parquet(fetch('interactions/train.parquet'))
test     = pl.read_parquet(fetch('interactions/test.parquet'))
problems = pl.read_parquet(fetch('problems.parquet'))
users    = pl.read_csv(fetch('users.csv'))
tags_raw = pl.read_json(fetch('tags.json'))
tags = (
    tags_raw
    .transpose(include_header=True)
    .rename({"column": "tag_name", "column_0": "tag_index"})
    .with_columns(pl.col("tag_index").cast(pl.Int64))
)

train = train.with_columns(pl.col('final_rating').fill_null(0.0))
test  = test.with_columns(pl.col('final_rating').fill_null(0.0))
problems = problems.with_columns(pl.col('problem_rating').fill_null(0.0))

print(train.shape, test.shape, problems.shape, users.shape)

In [ ]:
SECONDS_PER_YEAR = 365.25 * 24 * 3600

def make_edge_features(df: pl.DataFrame) -> Tensor:
    solved = torch.tensor(df["solved"].to_list(), dtype=torch.float32)
    tries = torch.log1p(torch.tensor(df["tries"].to_list(), dtype=torch.float32)) / np.log(100.0)
    rating = torch.tensor(df["final_rating"].to_list(), dtype=torch.float32).clamp(0, 4000) / 4000.0
    exp = torch.tensor(df["experience_years"].to_list(), dtype=torch.float32).clamp(0, 10) / 10.0

    edge_df = (
        df.select(["user_index", "experience_years"])
        .join(users.select(["user_index", "registration_time_seconds"]), on="user_index", how="left")
        .with_columns(
            (
                pl.col("registration_time_seconds")
                + pl.col("experience_years") * SECONDS_PER_YEAR
            ).alias("submission_time_seconds") # miks ma seda alguses ei salvestanud? :(
        )
    )

    latest_by_user = (
        edge_df
        .group_by("user_index")
        .agg(pl.max("submission_time_seconds").alias("latest_submission_time_seconds"))
    )

    edge_df = edge_df.join(latest_by_user, on="user_index", how="left").with_columns(
        (
            pl.col("latest_submission_time_seconds") - pl.col("submission_time_seconds")
        ).alias("time_diff_seconds")
    )

    recency = 1.0 / (1.0 + torch.tensor(edge_df["time_diff_seconds"].to_list(), dtype=torch.float32).clamp_min(0.0) / SECONDS_PER_YEAR)
    return torch.stack([solved, tries, rating, exp, recency], dim=1)  # (E, 5)
    
train_edge_feat = make_edge_features(train).to(device)
test_edge_feat = make_edge_features(test).to(device)

In [ ]:
u_idx = torch.tensor(train['user_index'].to_list(), dtype=torch.long)
p_idx = torch.tensor(train['problem_index'].to_list(), dtype=torch.long)

n_users = len(users)
n_problems = len(problems)
edge_index = torch.stack([
    torch.cat([ u_idx+n_problems, p_idx ]),
    torch.cat([ p_idx, u_idx+n_problems ])
])

data = Data(
    edge_index=edge_index,
    edge_attr=torch.cat([train_edge_feat, train_edge_feat], dim=0),
    num_nodes=n_users+n_problems
) 

loader = NeighborLoader(
    data,
    num_neighbors=[50, 30],
    batch_size=1024,
    input_nodes=torch.arange(n_problems, n_problems + n_users),
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [ ]:
from torch_scatter import scatter_sum, scatter_softmax

class UserEncoder(nn.Module):
    def __init__(self, edge_dim, prob_dim, hidden):
        super().__init__()
        self.msg_mlp = nn.Sequential(
            nn.Linear(edge_dim + prob_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden)
        )
        self.attn_mlp = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1)
        )

    def forward(self, prob_emb, edge_attr, user_index, u_emb_size):
        msg = self.msg_mlp(torch.cat([prob_emb, edge_attr], dim=-1)) 
        attn_logits = self.attn_mlp(msg).squeeze(-1)
        attn_weights = scatter_softmax(attn_logits, user_index, dim=0).unsqueeze(-1)
        user_emb = scatter_sum(msg*attn_weights, user_index, dim=0, dim_size=u_emb_size)
        return user_emb

class Scorer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(dim * 3, dim),
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Linear(dim // 2, 1)
        )

    def forward(self, u, p):
        u = F.normalize(u, dim=-1)
        p = F.normalize(p, dim=-1)
        x = torch.cat([u, p, u * p], dim=-1)
        return self.mlp(x).squeeze(-1)

class CFRecommender(nn.Module):
    def __init__(self, n_tags, edge_in, tag_dim=32, emb_dim=64, n_layers=2):
        super().__init__()
        
        self.tag_emb = nn.Embedding(n_tags, tag_dim)
        
        prob_in = 1 + tag_dim # rating + tags
        self.prob_encoder = nn.Sequential(
            nn.Linear(prob_in, emb_dim), nn.ReLU(),
            nn.Linear(emb_dim, emb_dim)
        )
        
        edge_out = 4*edge_in
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_in, 2*edge_in), nn.ReLU(),
            nn.Linear(2*edge_in, edge_out)
        )
        
        self.user_encoder = UserEncoder(edge_out, emb_dim, emb_dim)
        
        self.convs = nn.ModuleList([
            gnn.SAGEConv(emb_dim, emb_dim, root_weight=False) for _ in range(n_layers)
        ])
        
        self.scorer = Scorer(emb_dim)
        
        nn.init.xavier_uniform_(self.tag_emb.weight)

    def precompute_static_problem_features(self, problems_df):
        dev = next(self.parameters()).device
    
        self.ratings = torch.tensor(
            problems_df["problem_rating"].to_list(), dtype=torch.float32
        ).to(dev) / 4000.0
    
        tag_lists = problems_df["problem_tags"].to_list()
        flat_tags, counts = [], []
        for tags in tag_lists:
            if tags:
                flat_tags.extend(tags)
            counts.append(len(tags) or 1)
    
        self.flat_tags = torch.tensor(flat_tags, dtype=torch.long).to(dev)
        self.counts    = torch.tensor(counts, dtype=torch.float32).to(dev)
    
        tag_counts   = torch.tensor([len(t) for t in tag_lists], dtype=torch.long).to(dev)
        self.tag_ptr = torch.zeros(len(tag_lists) + 1, dtype=torch.long, device=dev)
        torch.cumsum(tag_counts, dim=0, out=self.tag_ptr[1:])
        
    def problem_features_subset(self, p_local):
        p_local = p_local.to(self.ratings.device)
        
        starts  = self.tag_ptr[p_local]
        ends    = self.tag_ptr[p_local + 1]
        lengths = ends - starts
    
        total        = lengths.sum().item()
        seg_starts   = starts.repeat_interleave(lengths)
        sub_flat_idx = torch.arange(total, device=self.flat_tags.device)
        offsets      = sub_flat_idx - lengths.cumsum(0).repeat_interleave(lengths) + seg_starts
        sub_tags     = self.flat_tags[offsets].long()
    
        n_local  = p_local.size(0)
        sub_rows = torch.arange(n_local, device=self.flat_tags.device).repeat_interleave(lengths)
    
        embeds   = self.tag_emb(sub_tags)
        tag_vecs = torch.zeros(n_local, self.tag_emb.embedding_dim, device=embeds.device)
        tag_vecs.scatter_add_(0, sub_rows.unsqueeze(1).expand_as(embeds), embeds)
        tag_vecs = tag_vecs / self.counts[p_local].unsqueeze(1)
    
        return torch.cat([self.ratings[p_local].unsqueeze(1), tag_vecs], dim=1)
        
    def encode(self, prob_feat, edge_index, edge_feat, u_emb_size):
        p_emb = self.prob_encoder(prob_feat)
        e_attr = self.edge_encoder(edge_feat)

        user_index = edge_index[0]
        problem_index = edge_index[1]
        
        u_emb = self.user_encoder(p_emb[problem_index], e_attr, user_index, u_emb_size)

        x = torch.cat([p_emb, u_emb], dim=0)
        n_prob = p_emb.size(0)
        bi_edge_index = torch.cat([
            torch.stack([ user_index + n_prob, problem_index ]),
            torch.stack([ problem_index, user_index + n_prob ]),
        ], dim=1)
        for conv in self.convs:
            x = x + conv(x, bi_edge_index)
            x = F.layer_norm(x, x.shape[-1:])
        return x[n_prob:], x[:n_prob]

    def score(self, u_emb, p_emb, u_idx, p_idx):
        return self.scorer(u_emb[u_idx], p_emb[p_idx])
    
    def bpr_loss(self, u_emb, p_emb, u_idx, pos_p, neg_p, margin=0.5):
        pos_score = self.score(u_emb, p_emb, u_idx, pos_p)
        neg_score = self.score(u_emb, p_emb, u_idx, neg_p)
        
        loss = F.softplus(-(pos_score - neg_score - margin)).mean()
        
        reg = (
            u_emb[u_idx].norm(2).pow(2) +
            p_emb[pos_p].norm(2).pow(2) +
            p_emb[neg_p].norm(2).pow(2)
        ) / u_idx.size(0)
        
        return loss + 1e-4 * reg
    

In [ ]:
model = CFRecommender(len(tags), train_edge_feat.size(1)).to(device)
model.precompute_static_problem_features(problems)

In [ ]:
class PopularityNegativeSampler:
    def __init__(self, p_idx_all, n_problems, temperature=0.75):
        counts     = torch.bincount(p_idx_all, minlength=n_problems).float()
        probs      = counts.pow(temperature)
        self.probs = probs / probs.sum()
        self.n_problems = n_problems

    @torch.no_grad()
    def sample(self, seed_global, seen_ptr, seen_cols, k=1):
        B = seed_global.size(0)
        cands = torch.multinomial(self.probs, B * 20, replacement=True).view(B, -1)
        result = cands[:, 0].clone()
        for i, u in enumerate(seed_global.tolist()):
            lo, hi = seen_ptr[u].item(), seen_ptr[u+1].item()
            seen_set = set(seen_cols[lo:hi].tolist())
            for c in cands[i].tolist():
                if c not in seen_set:
                    result[i] = c
                    break
        return result

class DNSNegativeSampler:
    def __init__(self, n_problems, emb_dim, refresh_every=200):
        self.cache         = torch.zeros(n_problems, emb_dim)
        self.refresh_every = refresh_every
        self.step          = 0
        self.n_problems    = n_problems

    @torch.no_grad()
    def refresh(self, model):
        model.eval()
        dev, chunk = next(model.parameters()).device, 4096
        embs = []
        for start in range(0, self.n_problems, chunk):
            p_local = torch.arange(start, min(start+chunk, self.n_problems), device=dev)
            embs.append(model.prob_encoder(model.problem_features_subset(p_local)).cpu())
        self.cache = F.normalize(torch.cat(embs), dim=-1)
        model.train()

    def maybe_refresh(self, model):
        self.step += 1
        if self.step % self.refresh_every == 0:
            self.refresh(model)

    @torch.no_grad()
    def sample_hard(self, u_emb, seed_global, seen_ptr, seen_cols, k=10, temperature=0.3):
        B      = u_emb.size(0)
        u_norm = F.normalize(u_emb.detach().cpu(), dim=-1)
        scores = u_norm @ self.cache.T  # [B, n_problems]
    
        rows, cols = [], []
        for i, u in enumerate(seed_global.tolist()):
            lo, hi = seen_ptr[u].item(), seen_ptr[u+1].item()
            if lo < hi:
                rows.append(torch.full((hi-lo,), i, dtype=torch.long))
                cols.append(seen_cols[lo:hi])
        if rows:
            scores[torch.cat(rows), torch.cat(cols)] = -1e9
    
        topk_scores, topk_idx = scores.topk(k, dim=1)  # [B, k]
    
        weights = F.softmax(topk_scores / temperature, dim=-1)  # [B, k]
        pick    = torch.multinomial(weights, num_samples=1).squeeze(1)  # [B]
        return topk_idx[torch.arange(B), pick]  # [B] CPU

In [ ]:
def build_seen_csr(train_df, n_users):
    user_idx = torch.tensor(train_df['user_index'].to_list(), dtype=torch.long)
    prob_idx = torch.tensor(train_df['problem_index'].to_list(), dtype=torch.long)
    order       = torch.argsort(user_idx)
    user_sorted = user_idx[order]
    prob_sorted = prob_idx[order]
    counts = torch.bincount(user_sorted, minlength=n_users)
    ptr    = torch.zeros(n_users + 1, dtype=torch.long)
    torch.cumsum(counts, dim=0, out=ptr[1:])
    return ptr, prob_sorted

seen_ptr, seen_cols = build_seen_csr(train, n_users)

In [ ]:
pop_sampler = PopularityNegativeSampler(p_idx, n_problems)
refresh_every = max(50, len(loader) // 8)
dns_sampler = DNSNegativeSampler(n_problems, emb_dim=64, refresh_every=refresh_every)
dns_sampler.refresh(model)

In [ ]:
def train_epoch(model, opt, loader, u_offset, neg_sampler, dns):
    model.train()
    total_loss = 0
    total_margin, total_frac = 0., 0.
    n_batches = 1

    for batch in tqdm(loader, desc='Train epoch'):
        batch       = batch.to(device)
        n_id        = batch.n_id
        ei          = batch.edge_index
        src_mask    = (n_id[ei[0]] >= u_offset) & (n_id[ei[1]] < u_offset)

        global_u    = n_id[ei[0][src_mask]] - u_offset
        global_p    = n_id[ei[1][src_mask]]
        edge_attr   = batch.edge_attr[src_mask]
        seed_global = batch.input_id - u_offset

        if global_u.numel() == 0:
            continue

        u_local, u_inv = torch.unique(
            torch.cat([global_u, seed_global]),
            return_inverse=True, sorted=True
        )
        u_inv_edges = u_inv[:len(global_u)]
        seed_local  = u_inv[len(global_u):]
        seed_cpu    = seed_global.cpu()

        lo       = seen_ptr[seed_cpu]
        hi       = seen_ptr[seed_cpu + 1]
        has_seen = (hi - lo) > 0
        
        pos_global = torch.zeros(seed_cpu.size(0), dtype=torch.long)
        if has_seen.any():
            lo_valid     = lo[has_seen]
            counts_valid = (hi[has_seen] - lo_valid)
            offsets      = torch.floor(
                torch.rand(counts_valid.size(0)) * counts_valid.float()
            ).long().clamp(max=counts_valid - 1)
            pos_global[has_seen] = seen_cols[lo_valid + offsets]
        
        keep_t      = has_seen.to(device)
        seed_local  = seed_local[keep_t]
        seed_global = seed_global[keep_t]
        seed_cpu    = seed_global.cpu()
        pos_global  = pos_global[has_seen] 

        neg_global = neg_sampler.sample(seed_cpu, seen_ptr, seen_cols) 

        p_local = torch.unique(
            torch.cat([global_p.cpu(), pos_global, neg_global]),
            sorted=True
        ).to(device)

        p_inv_edges = torch.searchsorted(p_local, global_p)
        encode_ei   = torch.stack([u_inv_edges, p_inv_edges])

        prob_feat        = model.problem_features_subset(p_local)
        u_emb, p_emb     = model.encode(prob_feat, encode_ei, edge_attr, len(u_local))

        dns.maybe_refresh(model)
        neg_global_dns = dns.sample_hard(
            u_emb[seed_local], seed_cpu, seen_ptr, seen_cols, k=5, temperature=0.1
        ).to(device)

        mix        = torch.rand(seed_local.size(0), device=device) > 0.3
        neg_global_final = torch.where(mix, neg_global_dns, neg_global.to(device))

        new_p = neg_global_final[~torch.isin(neg_global_final, p_local)]
        if new_p.numel() > 0:
            extra_feat = model.problem_features_subset(new_p)
            extra_emb  = model.prob_encoder(extra_feat)
            p_local, sort_idx = torch.cat([p_local, new_p]).sort()
            p_emb = torch.cat([p_emb, extra_emb], dim=0)[sort_idx]
        
        pos_local = torch.searchsorted(p_local, pos_global.to(device))
        neg_local = torch.searchsorted(p_local, neg_global_final)

        with torch.no_grad():
            ps = model.score(u_emb, p_emb, seed_local, pos_local)
            ns = model.score(u_emb, p_emb, seed_local, neg_local)
            total_margin += (ps - ns).mean().item()
            total_frac   += (ps > ns).float().mean().item()
            n_batches    += 1

        assert pos_local.max() < p_emb.size(0), f"pos oob: {pos_local.max()} >= {p_emb.size(0)}"
        assert neg_local.max() < p_emb.size(0), f"neg oob: {neg_local.max()} >= {p_emb.size(0)}"
        assert seed_local.max() < u_emb.size(0), f"seed oob: {seed_local.max()} >= {u_emb.size(0)}"
        loss = model.bpr_loss(u_emb, p_emb, seed_local, pos_local, neg_local)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total_loss += loss.item()

    print(f"margin={total_margin/n_batches:.3f}  frac_correct={total_frac/n_batches:.3f}")
    return total_loss / len(loader)

In [ ]:
@torch.no_grad()
def top_k_recommendations(model, history, n_problems, k=10, hide_seen=True):
    model.eval()
    device = next(model.parameters()).device

    seen_p = torch.tensor(history['problem_index'].to_list(), dtype=torch.long, device=device)
    ea     = make_edge_features(history).to(device)

    all_p  = torch.arange(n_problems, device=device)

    seen_local = torch.searchsorted(all_p, seen_p)
    ei = torch.stack([
        torch.zeros(seen_p.numel(), dtype=torch.long, device=device),  # user 0
        seen_local
    ])

    prob_feat    = model.problem_features_subset(all_p)
    u_emb, p_emb = model.encode(prob_feat, ei, ea, u_emb_size=1)
    # u_emb: [1, D],  p_emb: [n_problems, D]

    scores = model.scorer(
        u_emb.expand(n_problems, -1),   # [n_problems, D]
        p_emb                            # [n_problems, D]
    )

    if hide_seen:
        scores[seen_p] = -1e9

    return torch.topk(scores, k).indices.cpu().tolist()

In [ ]:
@torch.no_grad()
def evaluate(model, test_df, n_problems, n_users, k=10, min_interactions=100, sample_size=1000):
    model.eval()
    grouped = (
        test_df
        .group_by("user_index")
        .agg([pl.col("problem_index").alias("items"), pl.len().alias("n_interactions")])
        .filter(pl.col("n_interactions") > min_interactions)
        .sample(n=sample_size, seed=42)
    )

    # ── build all ctx/tgt splits upfront ────────────────────────────────────
    ctx_rows_list, tgt_sets = [], []
    rng = np.random.default_rng(42)
    for row in grouped.iter_rows(named=True):
        items = np.array(row['items'])
        perm  = rng.permutation(len(items))
        split = max(1, len(items) // 2)
        ctx_idx = items[perm[:split]]
        tgt_sets.append(set(items[perm[split:]].tolist()))
        ctx_rows_list.append(
            test_df.filter(
                (pl.col("user_index") == row["user_index"]) &
                (pl.col("problem_index").is_in(ctx_idx.tolist()))
            )
        )

    # ── one batched forward pass for ALL eval users ──────────────────────────
    all_u, all_p, all_attr = [], [], []
    for local_uid, ctx in enumerate(ctx_rows_list):
        p_idx_ctx = torch.tensor(ctx['problem_index'].to_list(), dtype=torch.long)
        attr      = make_edge_features(ctx).to(device)
        all_u.append(torch.full((len(p_idx_ctx),), local_uid, dtype=torch.long))
        all_p.append(p_idx_ctx)
        all_attr.append(attr)

    all_u    = torch.cat(all_u).to(device)
    all_p    = torch.cat(all_p).to(device)
    all_attr = torch.cat(all_attr)

    p_local, p_inv = torch.unique(all_p, return_inverse=True, sorted=True)
    encode_ei      = torch.stack([all_u, p_inv])

    prob_feat    = model.problem_features_subset(p_local)
    u_emb, _     = model.encode(prob_feat, encode_ei, all_attr, len(ctx_rows_list))
    # u_emb: [n_eval_users, D]

    # ── encode all problems once ─────────────────────────────────────────────
    all_p_emb = []
    for start in range(0, n_problems, 4096):
        end   = min(start + 4096, n_problems)
        p_loc = torch.arange(start, end, device=device)
        all_p_emb.append(model.prob_encoder(model.problem_features_subset(p_loc)))
    all_p_emb = F.normalize(torch.cat(all_p_emb), dim=-1)  # [n_problems, D]
    u_emb_n   = F.normalize(u_emb, dim=-1)                 # [n_eval_users, D]

    # ── score matrix + mask seen ─────────────────────────────────────────────
    scores = u_emb_n @ all_p_emb.T                         # [n_eval_users, n_problems]
    scores[all_u, all_p] = -1e9                            # mask ctx problems

    _, topk = scores.topk(k, dim=1)                        # [n_eval_users, k]
    recs    = topk.cpu().tolist()

    # ── metrics ──────────────────────────────────────────────────────────────
    ndcg_list, recall_list = [], []
    for recommendations, tgt_set in zip(recs, tgt_sets):
        hits   = sum(p in tgt_set for p in recommendations)
        recall_list.append(hits / max(len(tgt_set), 1))
        dcg    = sum(1.0/math.log2(i+2) for i,p in enumerate(recommendations) if p in tgt_set)
        idcg   = sum(1.0/math.log2(i+2) for i in range(min(len(tgt_set), k)))
        ndcg_list.append(dcg / idcg if idcg > 0 else 0.0)

    n = len(recall_list)
    return {
        f"ndcg":   sum(ndcg_list)  / n if n else 0.0,
        f"recall": sum(recall_list) / n if n else 0.0,
        "n_users": n,
    }

In [ ]:
def get_scheduler(opt, n_epochs, warmup_epochs=3):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / warmup_epochs          # linear warmup
        progress = (epoch - warmup_epochs) / (n_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))   # cosine decay
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

N_EPOCHS  = 30
opt       = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = get_scheduler(opt, N_EPOCHS, warmup_epochs=3)
best_ndcg = 0.0
k = 50

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    avg_loss = train_epoch(model, opt, loader, n_problems, pop_sampler, dns_sampler)
    scheduler.step()
    metrics = evaluate(model, test, n_problems, n_users, k)
    mark = ''
    if metrics['ndcg'] > best_ndcg:
        best_ndcg = metrics['ndcg']
        torch.save(model.state_dict(), MODEL_PATH)
        mark = ' ← best'
    print(f"E{epoch:02d} loss={avg_loss:.4f}  ndcg@{k}={metrics['ndcg']:.4f}  recall@{k}={metrics['recall']:.4f}  n_eval={metrics['n_users']}  [{time.time()-t0:.1f}s]{mark}")

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))

In [ ]:
import requests

def fetch_user_history(handle: str, user_index: int):
    base_url = 'https://codeforces.com/api/'
    
    info_res = requests.get(f'{base_url}user.info?handles={handle}')
    info_res.raise_for_status()
    user_info = info_res.json()['result'][0]
    
    current_rating = user_info.get('rating', 0)
    reg_ts = user_info['registrationTimeSeconds']
    
    rating_res = requests.get(f'{base_url}user.rating?handle={handle}')
    rating_res.raise_for_status()
    rating_history = rating_res.json().get('result', [])
    
    status_res = requests.get(f'{base_url}user.status?handle={handle}')
    status_res.raise_for_status()
    submissions = status_res.json().get('result', [])
    
    handled_verdicts = {
        "FAILED", "OK", "COMPILATION_ERROR", "RUNTIME_ERROR", 
        "WRONG_ANSWER", "TIME_LIMIT_EXCEEDED", "MEMORY_LIMIT_EXCEEDED", 
        "IDLENESS_LIMIT_EXCEEDED", "SECURITY_VIOLATED", "REJECTED"
    }
    
    problem_subs = {}
    for sub in submissions:
        reg_ts = min(reg_ts, sub.get('creationTimeSeconds', float('inf')))
        if sub.get("verdict") not in handled_verdicts:
            continue
        if sub["problem"]["type"] != "PROGRAMMING":
            continue
            
        p = sub["problem"]
        pid = f"{p.get('contestId', '')}.{p.get('index', '')}"
        problem_subs.setdefault(pid, []).append(sub)
        
    history = []
    for pid, subs in problem_subs.items():
        dataset_row = problems.filter(pl.col('problem_id') == pid)
        if dataset_row.is_empty():
            continue
        problem_index = dataset_row[0, 'problem_index']

        subs_chrono = list(reversed(subs))
        verdicts = [s.get("verdict") for s in subs_chrono]
        
        solved = "OK" in verdicts
        first_ok_idx = verdicts.index("OK") if solved else None
        tries = (first_ok_idx + 1) if solved else len(subs_chrono) # type: ignore
        
        decisive_sub = subs_chrono[first_ok_idx] if solved else subs_chrono[-1] # type: ignore
        decisive_ts = decisive_sub.get("creationTimeSeconds", reg_ts)
        
        historical_rating = None
        best_ts = -1
        for entry in rating_history:
            t_hist = entry.get("ratingUpdateTimeSeconds", 0)
            if t_hist <= decisive_ts and t_hist > best_ts:
                best_ts = t_hist
                historical_rating = entry.get("newRating")
        
        history.append({
            "problem_index": problem_index,
            "solved": solved,
            "tries": tries,
            "final_rating": historical_rating or 0,
            "experience_years": (decisive_ts - reg_ts) / (365.25 * 24 * 3600),
            "user_index": user_index
        })
        
    return pl.DataFrame(history), reg_ts

def get_problem_url(contest_id, index):
    if not contest_id:
        return f"https://codeforces.com/problemsets/acmsguru/problem/99999/{index}"
    if int(contest_id) >= 100000:
        return f"https://codeforces.com/gym/{contest_id}/problem/{index}"
    else:
        return f"https://codeforces.com/contest/{contest_id}/problem/{index}"

In [ ]:
handle = 'tourist'

existing_user = users.filter(pl.col('handle') == handle)
if len(existing_user) > 0:
    user_index = existing_user['user_index'][0]
else:
    user_index = n_users
history, reg_ts = fetch_user_history(handle, user_index)
if len(existing_user) == 0:
    users = users.vstack(pl.DataFrame({ "handle": handle, "user_index": user_index, "registration_time_seconds": reg_ts }))

recs = top_k_recommendations(model, history, n_problems)
for idx in recs:
    pid = problems.filter(pl.col('problem_index') == idx).select(pl.first('problem_id'))['problem_id'][0]
    print(get_problem_url(*pid.split('.')))